# SP519 · Prédiction du flux d'électrons de 1 MeV en LEO (NOAA-15)

**Objectif** : prédire le flux d'électrons ~1 MeV mesuré par NOAA-15 / SEM-2 (canal `mep_pro_tel90_flux_p6`)
à +12, +24, +36 et +48 h à partir du vent solaire (OMNI), en régression et en classification d'« événements ».

## Step 1 : préparation des données et analyse statistique
1. Paramètres et imports
2. Flux d'électrons NOAA-15 : ce que mesure P6, téléchargement et réduction
3. Paramètres solaires et géophysiques (OMNI horaire)
4. Nettoyage : contamination par les protons solaires (SEP), couverture, bruit de fond
5. Choix de la région de la ceinture (plage de L)
6. Moyennes sur 12 h et synchronisation → table finale
7. Visualisation des séries temporelles
8. Statistiques descriptives
9. Relations flux ↔ paramètres solaires
10. Valeurs aberrantes
11. Seuil d'événement (95ᵉ percentile)
12. Export de la table 12 h

Le notebook s'exécute de haut en bas sur un noyau neuf.

## 1. Paramètres et imports

In [ ]:
import os
import re
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import h5py      # les fichiers NOAA-15 sont en netCDF-4 = HDF5
import cdflib    # les fichiers OMNI sont en CDF
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'lines.linewidth': 1.0, 'font.size': 9})

# Dossiers (relatifs au notebook)
PROJ = os.getcwd()
DATA = os.path.join(PROJ, 'data')
FIG = os.path.join(PROJ, 'figures')
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Paramètres de l'étude
START, END = pd.Timestamp('2020-01-01'), pd.Timestamp('2025-01-01')   # période [START, END[
STEP = '12h'            # résolution temporelle des prédictions
L_LO, L_HI = 4.0, 6.0   # région de la ceinture retenue (justifiée en section 5)
P_EVENT = 95            # percentile définissant un "événement"

def savefig(fig, name):
    # Sauvegarde des figures pour le rapport
    fig.savefig(os.path.join(FIG, name), dpi=150, bbox_inches='tight')

print('Dossier du projet :', PROJ)